In [1]:
# Load necessary modules
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib
from matplotlib import pyplot as plt
import matplotlib.backends.backend_pdf
from Bio import SeqIO
import glob, pickle, os, re, subprocess, csv, sys, io
from contextlib import redirect_stdout

from uuid import uuid4
import argparse
from tqdm import tqdm 
import time

sys.path.append(os.path.abspath('/global/scratch/projects/fc_mvslab/predictors'))
from adpred import ADpred as adp
# from PADDLE import paddle
from iupred3.iupred import get_iupred

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, Flatten, Dense, Dropout, Activation
from tensorflow.keras import regularizers
from tensorflow.keras.activations import softplus
from tensorflow.keras import backend as K
from tensorflow.keras import metrics

# Limit tensorflow thread usage
import os
import ipyparallel as ipp

# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
# os.environ["TF_NUM_INTEROP_THREADS"] = "1"


    
If you wish to use a local installation of psipred, please indicate this,
    by assigning the path to local_psipred (e.g. local_psipred = "~/psipred/run_psipred")




In [6]:
# Goal: Troubleshoot adpred for file 87

In [42]:
fasta_name = "../data/interpro_uniprot_evid_1_2_split/pt_77.fasta"
output_dir = "../output/interpro_uniprot_evid_1_2_dask/"

In [43]:
print("Loading sequences")
aa_lst = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

fasta_name_tmp = fasta_name.split('/')[-1].removesuffix('.fasta').removesuffix('.fa')
recs = list(SeqIO.parse(fasta_name, 'fasta'))
fasta_name = fasta_name_tmp

seqs_to_keep = []
for r in recs:
    seq_str = str(r.seq)

    # Filtering sequences with too few AAs and non-standard AAs
    if (len(seq_str) < 30):
        print("Removing", seq_str, "because it is too short")
        continue
    elif (len(set(seq_str) - set(aa_lst)) > 0):
        print("Removing",seq_str, "because it contains non-standard amino acids")
        continue # No need for 
    else:
        seqs_to_keep.append(r)

recs = seqs_to_keep
    

Loading sequences


In [44]:
str(recs[710].seq)

'MAPPVQSQRLPPPSQPSGKGEVGELKTQLRQLAGSRAAGAEDLKRELYKKVISYMTVGIDVSSLFSEMVMCSATSDIVLKKMCYLYVGNYAKVHPDLALLTINFLQKDCQDEDPMIRGLALRSLCSLRVKNLVEYLVGPLRTGLQDSNSYVRTVAAIGALKLYHISPSTCIDSDFPVALKALMMHDPDAKVVASCLSSLQEILTLEGNASEEASKERELLLSKPVIYFLLNRIREFSEWAQCLVLDLVSKYVPSENSEIFDIMNLLEDRLQHANSAVVLATINVFLHLTISMADVHQQVYERIKAPLLTLVGSGSSEQSYAVLSHLHLLVMRAPILFSSDYKHFYCRYSDPSYVKKLKLEMLTAVANESNTYEIVTELSEYAANVDVLIARESIRAVGKIALQQYDVNAIVDRLLQFLEMEKDYVTAETLVLVKDLLRKYPQWSQDCIAVVGNVSSKNVEEPKAKAALVWMLGEYSQDMLDAPYILESLIDSWDEEDSAEVRLHLLTAVAKCFFKRPPETQKALGAALAAGLADAHQDVHDRALFYYRLLQHDVSVAERVINPPKQAVSVFADTQSSEVKDRIFDEFNSLSVIYQEPSYMFIDKEHRGPFEFSEDIGALATGASPDNSVLPSQRVEASDNDLLLSASEKEENRGIAANGSLAYSAPDYNGSSTALVSHSPSPTANTSAASGAPSTASISQPSIEIDDLLGLGPVTTPTPIPALKLNSKIVLDPATFQRKWGQLPIALTQDCSLTAQGIASLTSPQVLLRHMQGHLIQCIASGGQPPNFKFFFFAQKSEAPSGAGFFLVECLVNTSSSKAHVKVKADEASLSQDFAALFQSALLKFGVS'

In [25]:
def return_adpred_ss(r, output_dir):
    sequence = r.seq
    sub_fasta_name = re.sub(r'\s+', '_', r.id)
    computed="{}/ss/{}/{}.ss2".format(output_dir, sub_fasta_name, sub_fasta_name)
    df = pd.read_table(computed, 
                       skiprows=[0], sep='\s+', names=['position', 'residue', 'ss', 'c', 'h', 'e'])
    return ''.join(df.ss.values).replace('C','-')

In [26]:
return_adpred_ss(recs[711], output_dir)

'-------------HHH-------------------------HHH----------------------------------------------------------------------------HHHHHHHHHH--------HHHHHHHHHHH-HHHHH----HHHHHHHH-HHHH-EEEE-----------EEEE-HHHHHHH-----EE-------------HHHH---------------------------------------------------HHHHHHHH---'

In [28]:
adpred_ss_all = []
seqs = []
ids = []

for r in recs:
    adpred_ss_all.append(return_adpred_ss(r, output_dir))
    seqs.append(r.seq)
    ids.append(r.id)
    
ss_results = pd.DataFrame({"id" : ids, "seq" : seqs, "ss" : adpred_ss_all})
ss_results

,id,seq,ss
0,tr|A0A8G0VU03|A0A8G0VU03_9ROSI,"(M, S, N, S, Q, P, Q, N, T, S, T, Q, P, H, D, ...",------------------------HHH---------------HHH-...
1,tr|A0A8I3B0M7|A0A8I3B0M7_PONAB,"(M, N, D, F, G, I, K, N, M, D, Q, V, A, P, V, ...",--------HHHHH--HH------------------HHH--------...
2,tr|A0A8I3WJI5|A0A8I3WJI5_CALJA,"(M, L, D, P, K, C, D, R, K, R, A, R, E, F, I, ...",-------HHHHHHHH-----HHHH----------HHHHHHHHHHHH...
3,tr|A0A8J8YT14|A0A8J8YT14_CALJA,"(M, A, I, T, L, Q, P, S, D, L, I, F, E, F, A, ...",---EE----EEEEEE------HHHH--------EEEE-------HH...
4,tr|A0A8K0ZUK2|A0A8K0ZUK2_CHICK,"(M, A, G, A, R, D, P, L, R, R, P, P, R, P, P, ...",----------------------------------------------...
...,...,...,...
995,tr|F7HBY1|F7HBY1_CALJA,"(M, G, P, V, M, P, P, S, K, K, P, E, S, S, G, ...",----------------EE----------------------------...
996,tr|F7HFL8|F7HFL8_MACMU,"(M, M, Q, E, S, G, T, E, T, K, S, N, G, S, A, ...",------------------------HH----------------HHH-...
997,tr|F7HH72|F7HH72_CALJA,"(M, G, P, V, M, P, P, S, K, K, P, E, S, S, G, ...",----------------------------------------------...
998,tr|F7HIN3|F7HIN3_MACMU,"(M, C, D, R, N, G, G, R, R, L, R, Q, W, L, I, ...",----------HHHHHHHHHH-----EEEE----EEEEEE-------...


In [29]:
ss_results[ss_results["ss"].isna()]

,id,seq,ss


In [30]:
ss_results["ss"].value_counts()

ss
---------------EEEEEE--EEEE------EEE--EEEEEEEEEE---------HHHEEEEEE-----------EEEEEE------------------EEEEEEE---HHHHHHHHHHHHHH---HHH---------------EE--HHHHHHHHHHHHHHH--HHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHH-------HHHHHHHHHHH--------------HHHHHHHHHHHHHHHHH-HHHH----EEHHHHHHHHHHH-------HHHHHHHHHHHHH----EEEEEE----EEEE----HHHHHHHHHHHHHH-----HHHHHHHH-HHHHHHHHHHHHHHHH-----------------------                                                                                                                                                                         2
----HHHHHHHHHHHHH---EEE---------HHH-------HHHHHHHHHHHH---EEEEEEHHHHHHHHHHHHHHHHHHHH---HHH--HHHH--------------------------------HHH------------------------EE---------------                                                                                                                                                                                                                                                                                   

In [32]:
def make_pred(r):
    sequence = r.seq
    
    adp_prot = adp.protein('x', sequence)
    # Load pre-computed secondary structure predicted by PSIPRED V4.
    adp_prot.second_struct = return_adpred_ss(r, output_dir)
    # Run ADpred
    adp_prot.predict()
    adpred_preds = adp_prot.predictions
    return adpred_preds

In [33]:
%%time
results = make_pred(recs[711])
results

1/1 [==============================] - 0s 15ms/step
CPU times: user 9.42 s, sys: 186 ms, total: 9.6 s
Wall time: 17.5 s


array([8.92296666e-04, 8.65946349e-04, 1.04481389e-03, 6.79613615e-04,
       6.73374045e-04, 6.61538914e-04, 4.12585068e-04, 4.66680329e-04,
       5.71319833e-04, 5.31609694e-04, 3.28547932e-04, 6.40054874e-04,
       4.17759555e-04, 7.16565584e-04, 7.43842160e-04, 1.22487533e-03,
       7.77626468e-04, 5.42941329e-04, 4.88006190e-04, 5.44368348e-04,
       1.12039060e-03, 1.52285653e-03, 1.32968149e-03, 1.00295758e-03,
       4.24980972e-04, 4.30522399e-04, 4.59961389e-04, 4.46418300e-04,
       6.76790369e-04, 6.89300126e-04, 3.16023856e-04, 2.25990501e-04,
       1.18492586e-04, 1.48584077e-04, 8.49862336e-05, 1.34160058e-04,
       2.58141023e-04, 2.86448776e-04, 2.49136792e-04, 2.26908756e-04,
       2.86289054e-04, 2.57982552e-04, 1.92943291e-04, 2.81025394e-04,
       3.51707247e-04, 3.31405929e-04, 3.51686118e-04, 3.04041430e-04,
       3.33066593e-04, 2.08364712e-04, 1.54508583e-04, 1.96754947e-04,
       1.58642113e-04, 1.33421418e-04, 1.04580969e-04, 6.99872035e-05,
      

In [41]:
# Splitting the file into two
 
input_file = "../data/interpro_uniprot_evid_1_2_split/pt_87.fasta"
output_file1 = "../data/interpro_uniprot_evid_1_2_split/pt_87_1.fasta"
output_file2 = "../data/interpro_uniprot_evid_1_2_split/pt_87_2.fasta"

# Read all records
records = list(SeqIO.parse(input_file, "fasta"))

# Sanity check
assert len(records) == 1000, f"Expected 1000 records, found {len(records)}"

# Split
part1 = records[:500]
part2 = records[500:]

# Write outputs
SeqIO.write(part1, output_file1, "fasta")
SeqIO.write(part2, output_file2, "fasta")

print("Split complete!")


Split complete!


---

In [28]:
# ipp.__version__   # check that at least version 7
# mycluster = ipp.Cluster(n = int(os.getenv('SLURM_CPUS_ON_NODE')))
# mycluster.start_cluster_sync()

# c = mycluster.connect_client_sync()
# c.wait_for_engines(n = int(os.getenv('SLURM_CPUS_ON_NODE')))
# c.ids

In [29]:
# dview = c[:]
# dview.block = True

# # Load necessary modules on workers
# dview.execute('import sys, os')
# dview.execute('import pandas as pd')
# dview.execute('import numpy as np')
# dview.execute('import re, subprocess')
# dview.execute('import Bio')
# dview.execute('from adpred import ADpred as adp')

In [ ]:
%%time

# Push PSIPRED output to workers
dview.push(dict( ss_dict=ss_dict ))

# Need a wrapper function because map() only operates on one argument
def wrapper(r):
    seq = str(r.seq)
    prot = adp.protein('x', seq)
    
    # Load horizontal secondary structure data
    prot.second_struct = ss_dict[seq]
    
    # Run ADpred
    prot.predict()
    return (prot.sequence, prot.predictions)


# Run a parallel map, executing the wrapper function on indices 0,...,n-1
lview = c.load_balanced_view()
# Cause execution on main process to wait while tasks sent to workers finish
lview.block = True 
out = lview.map(wrapper, recs)   # Run calculation in parallel
adpred_out = dict(out)

In [18]:
%%time
results = make_pred(recs[0])
results

1/1 [==============================] - 0s 15ms/step
CPU times: user 16.6 s, sys: 280 ms, total: 16.9 s
Wall time: 23.3 s


array([5.61811103e-05, 4.40443437e-05, 6.13181037e-05, 8.26251780e-05,
       2.52670958e-04, 1.78079674e-04, 1.09058812e-04, 1.21632787e-04,
       1.71658932e-04, 1.93006039e-04, 8.04404786e-04, 6.22320047e-04,
       6.45624357e-04, 1.80142326e-03, 5.63441950e-04, 4.95417858e-04,
       3.62068851e-04, 1.36827282e-03, 1.21765968e-03, 1.34220789e-03,
       1.34051521e-03, 1.32528564e-03, 1.82191841e-03, 2.72152107e-03,
       2.81181047e-03, 1.71899411e-03, 6.05565356e-03, 4.46838839e-03,
       1.06152566e-02, 8.11810512e-03, 4.71947761e-03, 3.67023260e-03,
       7.94061285e-04, 2.81579996e-04, 2.09606311e-04, 4.90649196e-04,
       6.99731114e-04, 8.65662529e-04, 2.65891111e-04, 1.42347824e-04,
       9.49105524e-05, 1.07035354e-04, 1.69697712e-04, 2.37224420e-04,
       3.86559317e-04, 1.62812765e-04, 2.11327089e-04, 1.26290659e-04,
       1.67787075e-04, 2.36757114e-04, 1.56015085e-04, 8.74015896e-05,
       2.28480130e-05, 3.43243846e-05, 6.78522483e-05, 8.36226463e-05,
      

In [23]:
dict([("seq", results), ("seq2", results)])

{'seq': array([5.61811103e-05, 4.40443437e-05, 6.13181037e-05, 8.26251780e-05,
        2.52670958e-04, 1.78079674e-04, 1.09058812e-04, 1.21632787e-04,
        1.71658932e-04, 1.93006039e-04, 8.04404786e-04, 6.22320047e-04,
        6.45624357e-04, 1.80142326e-03, 5.63441950e-04, 4.95417858e-04,
        3.62068851e-04, 1.36827282e-03, 1.21765968e-03, 1.34220789e-03,
        1.34051521e-03, 1.32528564e-03, 1.82191841e-03, 2.72152107e-03,
        2.81181047e-03, 1.71899411e-03, 6.05565356e-03, 4.46838839e-03,
        1.06152566e-02, 8.11810512e-03, 4.71947761e-03, 3.67023260e-03,
        7.94061285e-04, 2.81579996e-04, 2.09606311e-04, 4.90649196e-04,
        6.99731114e-04, 8.65662529e-04, 2.65891111e-04, 1.42347824e-04,
        9.49105524e-05, 1.07035354e-04, 1.69697712e-04, 2.37224420e-04,
        3.86559317e-04, 1.62812765e-04, 2.11327089e-04, 1.26290659e-04,
        1.67787075e-04, 2.36757114e-04, 1.56015085e-04, 8.74015896e-05,
        2.28480130e-05, 3.43243846e-05, 6.78522483e-05, 8

In [ ]:
adpred_ss = 